[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/onnx/tutorials/blob/main/07_ONNX_Runtime/04_Performance_Tuning/Performance_Tuning_Deep_Dive.ipynb)

# Performance Tuning — Deep Dive

A comprehensive exploration of ONNX Runtime performance tuning: optimization levels, threading,
profiling infrastructure, memory optimization, batch tuning, latency/throughput analysis,
tail latency, hardware considerations, and production best practices.

---

## Table of Contents

| # | Section | Focus |
|---|---------|-------|
| 1 | [Optimization Levels](#1-optimization-levels) | Graph transform aggressiveness |
| 2 | [Threading Deep Dive](#2-threading-deep-dive) | Intra-op, inter-op, Amdahl's Law |
| 3 | [Profiling Infrastructure](#3-profiling-infrastructure) | Built-in profiler, Chrome trace |
| 4 | [Memory Optimization](#4-memory-optimization) | Arenas, bandwidth, arithmetic intensity |
| 5 | [Batch Tuning](#5-batch-tuning) | Throughput and per-sample latency |
| 6 | [Latency vs Throughput](#6-latency-vs-throughput) | Operating point selection |
| 7 | [Tail Latency](#7-tail-latency) | p95/p99 analysis and reduction |
| 8 | [Hardware Counters](#8-hardware-counters) | Cache misses, NUMA, memory-bound ops |
| 9 | [Best Practices](#9-best-practices) | Production tuning checklist |
| 10 | [Case Study](#10-case-study) | End-to-end tuning walkthrough |

In [ ]:
# Install dependencies
!pip install onnxruntime onnx numpy matplotlib scipy -q

<a id='1'></a>
## 1. Performance Fundamentals

### Amdahl's Law

The theoretical maximum speedup from parallelization is bounded by the sequential fraction:

$$S(n) = \frac{1}{(1-p) + \frac{p}{n}}$$

Where:
- $S(n)$ = speedup with $n$ processors
- $p$ = fraction of work that is parallelizable
- $1-p$ = serial fraction (the bottleneck)

**Maximum achievable speedup** (infinite processors):

$$\lim_{n \to \infty} S(n) = \frac{1}{1-p}$$

If 90% of your model's computation is parallelizable ($p = 0.9$), the maximum speedup is $\frac{1}{0.1} = 10\times$, regardless of how many cores you add.

### Memory Bandwidth Model

Data movement often dominates inference latency:

$$B = \frac{\text{data\_size}}{t_{\text{transfer}}} \quad [\text{bytes/sec}]$$

For a tensor of shape $(B, C, H, W)$ in FP32:

$$\text{data\_size} = B \times C \times H \times W \times 4 \text{ bytes}$$

$$t_{\text{transfer}} = \frac{\text{data\_size}}{\text{BW}_{\text{effective}}}$$

### Roofline Model

The roofline model classifies workloads as compute-bound or memory-bound:

$$\text{Attainable FLOPS} = \min\left(\text{Peak FLOPS}, \text{BW}_{\text{mem}} \times \text{AI}\right)$$

Where **Arithmetic Intensity** (AI) is:

$$\text{AI} = \frac{\text{FLOPs}}{\text{Bytes transferred}} \quad [\text{FLOP/byte}]$$

The **ridge point** where compute meets bandwidth:

$$\text{AI}_{\text{ridge}} = \frac{\text{Peak FLOPS}}{\text{BW}_{\text{mem}}}$$

```
Performance                                           ┌── Peak FLOPS ──
(GFLOPS)    ╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱│
           ╱╱ Bandwidth-limited  ╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱│  Compute-limited
          ╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱│
         ╱                               ▲           │
        ╱                                │           │
       ╱                          Ridge Point        │
      ╱                                              │
     ╱                                               │
    ╱───────────────────────────────────────────────────────────────────
         Arithmetic Intensity (FLOP/byte) ──────────────────────────►
```

### Common Operations Classification

| Operation | AI (typical) | Classification |
|-----------|:-------------|:---------------|
| Elementwise (Relu, Add) | ~0.25 | Memory-bound |
| BatchNorm | ~1 | Memory-bound |
| Depthwise Conv | ~3-10 | Memory-bound |
| Dense Conv (3×3) | ~50-200 | Compute-bound |
| Large MatMul | ~100+ | Compute-bound |
| Softmax | ~3 | Memory-bound |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Visualize Amdahl's Law
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

n = np.arange(1, 65)
for p in [0.5, 0.75, 0.9, 0.95, 0.99]:
    speedup = 1 / ((1 - p) + p / n)
    ax1.plot(n, speedup, linewidth=2, label=f'p = {p} (max={1/(1-p):.1f}x)')

ax1.set_xlabel('Number of Processors (n)', fontsize=11)
ax1.set_ylabel('Speedup S(n)', fontsize=11)
ax1.set_title("Amdahl's Law: $S(n) = \\frac{1}{(1-p) + p/n}$", fontsize=12, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_xlim([1, 64])

# Roofline model
peak_flops = 200  # GFLOPS
bandwidth = 50    # GB/s
ridge = peak_flops / bandwidth

ai = np.logspace(-1, 3, 200)
roofline = np.minimum(peak_flops, bandwidth * ai)

ax2.loglog(ai, roofline, 'b-', linewidth=3, label='Roofline')
ax2.axvline(x=ridge, color='gray', linestyle='--', alpha=0.7, label=f'Ridge: AI={ridge:.1f}')

# Mark common operations
ops = [
    ('Relu', 0.25, 12),
    ('BN', 1.0, 40),
    ('Softmax', 3, 80),
    ('DW Conv', 8, 150),
    ('Conv 3x3', 80, 180),
    ('Large MatMul', 200, 195),
]
for name, ai_val, perf in ops:
    color = 'red' if ai_val < ridge else 'green'
    ax2.plot(ai_val, perf, 'o', markersize=10, color=color)
    ax2.annotate(name, (ai_val, perf), textcoords='offset points',
                xytext=(5, 5), fontsize=8)

ax2.fill_between(ai[ai < ridge], 0.1, bandwidth * ai[ai < ridge],
                alpha=0.08, color='red')
ax2.fill_between(ai[ai >= ridge], 0.1, peak_flops,
                alpha=0.08, color='green')
ax2.set_xlabel('Arithmetic Intensity (FLOP/byte)', fontsize=11)
ax2.set_ylabel('Performance (GFLOPS)', fontsize=11)
ax2.set_title('Roofline Model (CPU: 200 GFLOPS, 50 GB/s)', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3, which='both')
ax2.set_xlim([0.1, 1000])
ax2.set_ylim([1, 500])

plt.tight_layout()
plt.savefig('perf_fundamentals.png', dpi=150, bbox_inches='tight')
plt.show()

<a id='2'></a>
## 2. Profiling ORT Sessions

### Profiling Pipeline

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    ORT Profiling Workflow                                 │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                          │
│  Step 1: Enable                                                          │
│    so.enable_profiling = True                                            │
│    so.profile_file_prefix = "my_model"                                  │
│                                                                          │
│  Step 2: Execute                                                         │
│    session = InferenceSession(model, so)                                 │
│    for _ in range(N): session.run(...)  # N realistic runs               │
│                                                                          │
│  Step 3: Collect                                                         │
│    prof_file = session.end_profiling()                                   │
│                                                                          │
│  Step 4: Analyze                                                         │
│    Open in chrome://tracing                                              │
│    OR parse JSON programmatically                                        │
│                                                                          │
└─────────────────────────────────────────────────────────────────────────┘
```

### What the Profile Reveals

| Event Type | Information | Optimization Target |
|------------|-------------|--------------------|
| Node execution | Per-kernel time | Identify hotspot ops |
| Memory copy | H2D/D2H duration | Use IOBinding |
| Fence/sync | EP boundary waits | Reduce EP transitions |
| Memory alloc | Arena allocation time | Enable mem pattern |
| Session init | Optimization time | Cache optimized model |

### Profile Analysis Strategy

Apply the **80/20 rule**: typically 20% of operators account for 80% of execution time.

$$\text{Focus criterion}: \quad T_{\text{op}} > \frac{T_{\text{total}}}{5}$$

Any single operator consuming more than 20% of total time deserves dedicated optimization.

In [ ]:
import onnxruntime as ort
import onnx
from onnx import helper, TensorProto, numpy_helper
import numpy as np
import time
import json

# Build a multi-layer model for profiling
np.random.seed(42)
layer_sizes = [784, 512, 256, 128, 64, 10]
nodes = []
initializers = []

for i in range(len(layer_sizes) - 1):
    in_dim, out_dim = layer_sizes[i], layer_sizes[i+1]
    W = np.random.randn(in_dim, out_dim).astype(np.float32) * 0.01
    B = np.zeros(out_dim, dtype=np.float32)
    initializers.extend([
        numpy_helper.from_array(W, f"W{i}"),
        numpy_helper.from_array(B, f"B{i}"),
    ])
    
    in_name = "X" if i == 0 else f"H{i-1}r"
    nodes.append(helper.make_node("MatMul", [in_name, f"W{i}"], [f"H{i}"]))
    nodes.append(helper.make_node("Add", [f"H{i}", f"B{i}"], [f"H{i}b"]))
    if i < len(layer_sizes) - 2:
        nodes.append(helper.make_node("Relu", [f"H{i}b"], [f"H{i}r"]))
    else:
        nodes.append(helper.make_node("Softmax", [f"H{i}b"], ["Y"], axis=1))

X = helper.make_tensor_value_info("X", TensorProto.FLOAT, ["batch", 784])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, ["batch", 10])
graph = helper.make_graph(nodes, "DeepMLP", [X], [Y], initializer=initializers)
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
onnx.save(model, "perf_demo.onnx")

# Profile the model
so = ort.SessionOptions()
so.enable_profiling = True
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

sess = ort.InferenceSession("perf_demo.onnx", so, providers=["CPUExecutionProvider"])

x = np.random.randn(64, 784).astype(np.float32)
# Warmup
for _ in range(50):
    sess.run(None, {"X": x})
# Profiled runs
for _ in range(100):
    sess.run(None, {"X": x})

prof_file = sess.end_profiling()
print(f"Profile saved: {prof_file}")

# Analyze profile
with open(prof_file) as f:
    events = json.load(f)

kernel_times = {}
for ev in events:
    if isinstance(ev, dict) and ev.get('cat') == 'Node' and 'dur' in ev:
        name = ev['name']
        kernel_times[name] = kernel_times.get(name, 0) + ev['dur']

if kernel_times:
    total = sum(kernel_times.values())
    print(f"\n{'Kernel':<40} {'Time(μs)':<12} {'%':<8}")
    print("-" * 62)
    for name, dur in sorted(kernel_times.items(), key=lambda x: -x[1])[:8]:
        print(f"{name:<40} {dur:<12.0f} {dur/total*100:<8.1f}")
else:
    print(f"Found {len(events)} profile events")

import os
os.remove(prof_file)

<a id='3'></a>
## 3. Thread Tuning

### Two Dimensions of Parallelism

```
┌─────────────────────────────────────────────────────────────────────────┐
│                                                                          │
│  INTRA-OP PARALLELISM (within one operator)                             │
│                                                                          │
│  MatMul: C = A × B, where C ∈ ℝ^{M×N}                                 │
│                                                                          │
│  Thread 0: rows [0, M/4)      ┐                                        │
│  Thread 1: rows [M/4, M/2)    │ All threads work on                    │
│  Thread 2: rows [M/2, 3M/4)   │ the SAME operator                     │
│  Thread 3: rows [3M/4, M)     ┘                                        │
│                                                                          │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                          │
│  INTER-OP PARALLELISM (across independent operators)                    │
│                                                                          │
│  Graph DAG with independent branches:                                   │
│                                                                          │
│  Input ──┬── Branch A (Conv) ──┐                                       │
│          │                     ├── Concat ── Output                     │
│          └── Branch B (Pool) ──┘                                       │
│                                                                          │
│  Thread pool 0: executes Branch A  ┐ Different threads work on          │
│  Thread pool 1: executes Branch B  ┘ DIFFERENT operators simultaneously │
│                                                                          │
└─────────────────────────────────────────────────────────────────────────┘
```

### Optimal Thread Configuration

For a server with $N_c$ physical cores serving $W$ concurrent workers:

$$\text{intra\_op} = \left\lfloor \frac{N_c}{W} \right\rfloor$$

$$\text{inter\_op} = \begin{cases} 1 & \text{if model is sequential (CNN/Transformer)} \\ \min(W_{\text{graph}}, 4) & \text{if model has parallel branches} \end{cases}$$

Where $W_{\text{graph}}$ is the maximum width of the computation graph.

**Oversubscription penalty**: When total threads exceed physical cores:

$$\text{Overhead}_{\text{context\_switch}} = N_{\text{threads\_over}} \times T_{\text{switch}} \times f_{\text{switch}}$$

### NUMA Awareness

On multi-socket systems, memory access patterns matter:

$$T_{\text{memory}} = \begin{cases} T_{\text{local}} & \text{if data on same NUMA node} \\ T_{\text{remote}} \approx 1.5 \times T_{\text{local}} & \text{if cross-node access} \end{cases}$$

Bind ORT threads to a single NUMA node for consistent performance.

In [ ]:
import onnxruntime as ort
import numpy as np
import time

# Thread scaling experiment
thread_configs = [
    (1, 1), (2, 1), (4, 1), (8, 1),  # Vary intra-op
    (4, 1), (4, 2), (4, 4),            # Vary inter-op
]

x_test = np.random.randn(64, 784).astype(np.float32)
results = []

for intra, inter in thread_configs:
    so = ort.SessionOptions()
    so.intra_op_num_threads = intra
    so.inter_op_num_threads = inter
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    
    sess = ort.InferenceSession("perf_demo.onnx", so, providers=["CPUExecutionProvider"])
    
    # Warmup
    for _ in range(30):
        sess.run(None, {"X": x_test})
    
    # Benchmark
    times = []
    for _ in range(200):
        t0 = time.perf_counter()
        sess.run(None, {"X": x_test})
        times.append((time.perf_counter() - t0) * 1000)
    
    results.append({
        'intra': intra, 'inter': inter,
        'mean': np.mean(times), 'p50': np.median(times),
        'p95': np.percentile(times, 95), 'p99': np.percentile(times, 99),
        'std': np.std(times)
    })

print(f"{'Intra':>5} {'Inter':>5} {'Mean(ms)':>10} {'P50(ms)':>10} {'P95(ms)':>10} {'P99(ms)':>10} {'Std':>8}")
print("-" * 60)
for r in results:
    print(f"{r['intra']:>5} {r['inter']:>5} {r['mean']:>10.3f} {r['p50']:>10.3f} {r['p95']:>10.3f} {r['p99']:>10.3f} {r['std']:>8.3f}")

# Compute speedups relative to single-thread
base = results[0]['p50']
print(f"\nSpeedups (relative to intra=1, inter=1):")
for r in results:
    print(f"  intra={r['intra']}, inter={r['inter']}: {base/r['p50']:.2f}x")

<a id='4'></a>
## 4. Memory Optimization

### Memory Hierarchy and Latency

```
┌────────────────────────────────────────────────────────────────────────┐
│                    Memory Hierarchy (CPU)                               │
├────────────────────────────────────────────────────────────────────────┤
│                                                                        │
│  Registers    │  0 cycles    │  ~KB      │  Compiler-managed           │
│  ─────────────┼──────────────┼───────────┼─────────────────────────    │
│  L1 Cache     │  ~4 cycles   │  32-64KB  │  Per-core, data + instr    │
│  ─────────────┼──────────────┼───────────┼─────────────────────────    │
│  L2 Cache     │  ~12 cycles  │  256KB-1MB│  Per-core                  │
│  ─────────────┼──────────────┼───────────┼─────────────────────────    │
│  L3 Cache     │  ~40 cycles  │  8-64MB   │  Shared across cores       │
│  ─────────────┼──────────────┼───────────┼─────────────────────────    │
│  DRAM         │  ~200 cycles │  16-512GB │  Main memory               │
│  ─────────────┼──────────────┼───────────┼─────────────────────────    │
│  NVMe/SSD     │  ~100K cycles│  TB       │  Storage (model loading)   │
│                                                                        │
└────────────────────────────────────────────────────────────────────────┘
```

### ORT Memory Arena

The arena pre-allocates a contiguous memory block and suballocates from it:

$$T_{\text{arena\_alloc}} \approx O(1) \quad \text{vs} \quad T_{\text{malloc}} \approx O(\log n)$$

Arena configuration:

| Parameter | Effect | Default |
|-----------|--------|--------|
| `enable_cpu_mem_arena` | Use arena allocator | True |
| `enable_mem_pattern` | Record/replay alloc pattern | True |
| `enable_mem_reuse` | Share buffers for non-overlapping lifetimes | True |

### Peak Memory Estimation

For a model with $n$ intermediate tensors, each of size $s_i$, with lifetimes $[a_i, b_i]$:

$$\text{Peak memory} = \max_t \sum_{i: a_i \leq t \leq b_i} s_i$$

This is the **interval scheduling maximum overlap** — the minimum memory needed regardless of allocation strategy.

### Bandwidth Optimization

Reducing memory traffic through precision reduction:

| Precision | Bytes/element | Relative BW |
|-----------|:-------------|:------------|
| FP32 | 4 | 1.0x |
| FP16 | 2 | 2.0x |
| INT8 | 1 | 4.0x |

Effective bandwidth improvement from quantization:

$$\text{BW}_{\text{effective}} = \text{BW}_{\text{physical}} \times \frac{4}{\text{bytes\_per\_element}}$$

In [ ]:
import onnxruntime as ort
import numpy as np
import time

# Compare memory optimization settings
configs = [
    {"name": "Arena+Pattern", "arena": True, "pattern": True},
    {"name": "Arena only", "arena": True, "pattern": False},
    {"name": "No arena", "arena": False, "pattern": False},
]

x_test = np.random.randn(64, 784).astype(np.float32)

print(f"{'Config':<20} {'First(ms)':<12} {'Steady P50(ms)':<16} {'P99(ms)':<12}")
print("-" * 62)

for cfg in configs:
    so = ort.SessionOptions()
    so.enable_cpu_mem_arena = cfg["arena"]
    so.enable_mem_pattern = cfg["pattern"]
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    
    sess = ort.InferenceSession("perf_demo.onnx", so, providers=["CPUExecutionProvider"])
    
    # First run (pattern recording)
    t0 = time.perf_counter()
    sess.run(None, {"X": x_test})
    first_run = (time.perf_counter() - t0) * 1000
    
    # Warmup
    for _ in range(20):
        sess.run(None, {"X": x_test})
    
    # Steady state
    times = []
    for _ in range(300):
        t0 = time.perf_counter()
        sess.run(None, {"X": x_test})
        times.append((time.perf_counter() - t0) * 1000)
    
    print(f"{cfg['name']:<20} {first_run:<12.4f} {np.median(times):<16.4f} {np.percentile(times, 99):<12.4f}")

<a id='5'></a>
## 5. Graph Optimization Impact

### Optimization Passes and Their Effects

```
┌──────────────────────────────────────────────────────────────────┐
│                   Graph Optimization Pipeline                     │
├──────────────────────────────────────────────────────────────────┤
│                                                                  │
│  LEVEL 1: BASIC                                                  │
│  ├── Constant folding: eval static subgraphs at build time      │
│  ├── Dead code elimination: remove unreachable nodes            │
│  ├── Identity removal: X → Identity → Y  ⟹  X → Y             │
│  └── Redundant node elimination: duplicate computation          │
│                                                                  │
│  LEVEL 2: EXTENDED                                               │
│  ├── Conv + BN fusion: fold BN stats into Conv weights          │
│  ├── Conv + Relu/Clip: fused activation                         │
│  ├── MatMul + Add → Gemm: single BLAS call                     │
│  ├── Attention pattern fusion: Q/K/V → fused attention          │
│  └── Gelu approximation fusion                                  │
│                                                                  │
│  LEVEL 3: ALL (includes layout)                                  │
│  ├── NCHW → NHWC: better cache utilization on CPU               │
│  ├── Advanced attention: Flash Attention-like patterns           │
│  └── EP-specific optimizations                                   │
│                                                                  │
└──────────────────────────────────────────────────────────────────┘
```

### Fusion Memory Savings

For a fused `MatMul + Add + Relu` operating on tensors of size $S$:

- **Unfused**: Read $S$ (MatMul output) + Write $S$ + Read $S$ (Add) + Write $S$ + Read $S$ (Relu) + Write $S$ = $6S$ memory ops
- **Fused**: Read input + Read weight + Write $S$ = $\sim 2S$ memory ops

$$\text{Memory traffic reduction} = 1 - \frac{2S}{6S} = 66.7\%$$

For memory-bound operations, this translates directly to proportional speedup.

In [ ]:
import onnxruntime as ort
import numpy as np
import time

# Measure optimization impact systematically
x_test = np.random.randn(64, 784).astype(np.float32)

levels = [
    ("DISABLED", ort.GraphOptimizationLevel.ORT_DISABLE_ALL),
    ("BASIC", ort.GraphOptimizationLevel.ORT_ENABLE_BASIC),
    ("EXTENDED", ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED),
    ("ALL", ort.GraphOptimizationLevel.ORT_ENABLE_ALL),
]

level_results = []

for name, level in levels:
    so = ort.SessionOptions()
    so.graph_optimization_level = level
    so.intra_op_num_threads = 4
    
    # Measure creation time
    t0 = time.perf_counter()
    sess = ort.InferenceSession("perf_demo.onnx", so, providers=["CPUExecutionProvider"])
    create_ms = (time.perf_counter() - t0) * 1000
    
    # Warmup
    for _ in range(50):
        sess.run(None, {"X": x_test})
    
    # Benchmark
    times = []
    for _ in range(500):
        t0 = time.perf_counter()
        sess.run(None, {"X": x_test})
        times.append((time.perf_counter() - t0) * 1000)
    
    level_results.append({
        'name': name, 'create_ms': create_ms,
        'p50': np.median(times), 'p99': np.percentile(times, 99),
        'mean': np.mean(times)
    })

baseline = level_results[0]['p50']
print(f"{'Level':<12} {'Create(ms)':<12} {'P50(ms)':<10} {'P99(ms)':<10} {'Speedup':<10} {'Break-even N':<14}")
print("-" * 70)
for r in level_results:
    speedup = baseline / r['p50']
    # Break-even: N where extra creation cost is recouped
    extra_create = r['create_ms'] - level_results[0]['create_ms']
    savings_per_run = baseline - r['p50']
    breakeven = int(extra_create / savings_per_run) if savings_per_run > 0 else 0
    print(f"{r['name']:<12} {r['create_ms']:<12.3f} {r['p50']:<10.4f} {r['p99']:<10.4f} {speedup:<10.2f}x {breakeven:<14}")

<a id='6'></a>
## 6. IOBinding for GPU

### Transfer Overhead Analysis

For GPU inference, the host↔device transfer can dominate small-model latency:

$$T_{\text{H2D}} = T_{\text{launch}} + \frac{S_{\text{input}}}{\text{BW}_{\text{PCIe}}}$$

$$T_{\text{D2H}} = T_{\text{launch}} + \frac{S_{\text{output}}}{\text{BW}_{\text{PCIe}}}$$

Where:
- $T_{\text{launch}} \approx 5\text{-}10\mu s$ (DMA engine setup)
- $\text{BW}_{\text{PCIe Gen4 x16}} \approx 25$ GB/s

### Break-Even Analysis

IOBinding is worth implementing when transfer overhead is a significant fraction:

$$\frac{T_{\text{transfer}}}{T_{\text{total}}} > \theta \quad (\text{typically } \theta = 0.1)$$

### IOBinding API Pattern

```python
# Setup (once)
io_binding = session.io_binding()

# Per-inference (zero-copy if data already on device)
io_binding.bind_input('input', 'cuda', 0, np.float32, shape, data_ptr)
io_binding.bind_output('output', 'cuda', 0)  # ORT allocates on device
session.run_with_iobinding(io_binding)

# Get output (stays on device)
output_ortvalue = io_binding.get_outputs()[0]
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Model the benefit of IOBinding across different scenarios
input_sizes_mb = np.linspace(0.01, 10, 50)  # Input sizes in MB
compute_times_ms = [0.5, 2.0, 10.0, 50.0]  # Different model complexities

pcie_bw = 25  # GB/s = 25000 MB/s
launch_overhead = 0.01  # ms

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for t_compute in compute_times_ms:
    t_transfer = 2 * (launch_overhead + input_sizes_mb / (pcie_bw * 1000) * 1e6)  # ms, round-trip
    # Correction: input_sizes_mb in MB, BW in GB/s
    t_transfer = 2 * (launch_overhead + input_sizes_mb / 25)  # ms
    
    speedup = (t_compute + t_transfer) / t_compute
    ax1.plot(input_sizes_mb, speedup, linewidth=2, label=f'Compute = {t_compute}ms')

ax1.set_xlabel('Input Size (MB)', fontsize=11)
ax1.set_ylabel('IOBinding Speedup (×)', fontsize=11)
ax1.set_title('IOBinding Benefit vs Input Size\n(eliminating H2D + D2H)', fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.axhline(y=1.1, color='red', linestyle='--', alpha=0.5, label='10% threshold')

# Transfer fraction analysis
model_flops = np.logspace(7, 11, 50)  # 10M to 100B FLOPs
gpu_tflops = 20  # TFLOPS
input_mb = 1  # 1 MB input

t_compute_range = model_flops / (gpu_tflops * 1e12) * 1000  # ms
t_transfer_fixed = 2 * (0.01 + input_mb / 25)  # ms

transfer_fraction = t_transfer_fixed / (t_compute_range + t_transfer_fixed) * 100

ax2.semilogx(model_flops, transfer_fraction, 'b-', linewidth=2)
ax2.axhline(y=10, color='red', linestyle='--', label='10% threshold')
ax2.fill_between(model_flops, 10, transfer_fraction,
                where=transfer_fraction > 10, alpha=0.2, color='red',
                label='IOBinding beneficial')
ax2.set_xlabel('Model FLOPs', fontsize=11)
ax2.set_ylabel('Transfer Overhead (%)', fontsize=11)
ax2.set_title('When Transfer Overhead Dominates\n(1MB input, 20 TFLOPS GPU)', fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_ylim([0, 100])

plt.tight_layout()
plt.savefig('perf_iobinding.png', dpi=150, bbox_inches='tight')
plt.show()

<a id='7'></a>
## 7. Roofline Analysis for ORT Models

### Computing Arithmetic Intensity per Layer

For each layer type:

**MatMul** ($A \in \mathbb{R}^{M \times K}, B \in \mathbb{R}^{K \times N}$):
$$\text{FLOPs} = 2MKN$$
$$\text{Bytes} = (MK + KN + MN) \times \text{sizeof}(\text{dtype})$$
$$\text{AI}_{\text{MatMul}} = \frac{2MKN}{(MK + KN + MN) \times 4}$$

For square matrices ($M=K=N$): $\text{AI} = \frac{2N^3}{3N^2 \times 4} = \frac{N}{6}$

**Elementwise** (Relu, Add, etc.):
$$\text{FLOPs} = N_{\text{elements}}$$
$$\text{Bytes} = 2 \times N_{\text{elements}} \times 4 \quad (\text{read + write})$$
$$\text{AI}_{\text{elementwise}} = \frac{1}{8} = 0.125$$

**Softmax**:
$$\text{FLOPs} \approx 5N \quad (\text{max, sub, exp, sum, div})$$
$$\text{Bytes} = 2N \times 4$$
$$\text{AI}_{\text{softmax}} \approx \frac{5}{8} = 0.625$$

### Layer-wise Bottleneck Identification

```
Layer         AI        Ridge=4     Classification    Optimization
──────────────────────────────────────────────────────────────────
MatMul 784×512  ~65     > ridge     Compute-bound     More cores/SIMD
Add (bias)      0.125   < ridge     Memory-bound      Fuse with MatMul!
Relu            0.125   < ridge     Memory-bound      Fuse with previous!
MatMul 512×256  ~43     > ridge     Compute-bound     More cores/SIMD
Softmax         0.625   < ridge     Memory-bound      Flash attention
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Compute arithmetic intensity for our demo model layers
layers = [
    {"name": "MatMul 784×512", "flops": 2*784*512, "bytes": (784*512 + 512)*4,
     "type": "compute"},
    {"name": "Add bias (512)", "flops": 512, "bytes": 2*512*4,
     "type": "memory"},
    {"name": "Relu (512)", "flops": 512, "bytes": 2*512*4,
     "type": "memory"},
    {"name": "MatMul 512×256", "flops": 2*512*256, "bytes": (512*256 + 256)*4,
     "type": "compute"},
    {"name": "Add bias (256)", "flops": 256, "bytes": 2*256*4,
     "type": "memory"},
    {"name": "Relu (256)", "flops": 256, "bytes": 2*256*4,
     "type": "memory"},
    {"name": "MatMul 256×128", "flops": 2*256*128, "bytes": (256*128 + 128)*4,
     "type": "compute"},
    {"name": "MatMul 128×64", "flops": 2*128*64, "bytes": (128*64 + 64)*4,
     "type": "compute"},
    {"name": "MatMul 64×10", "flops": 2*64*10, "bytes": (64*10 + 10)*4,
     "type": "compute"},
    {"name": "Softmax (10)", "flops": 5*10, "bytes": 2*10*4,
     "type": "memory"},
]

# Multiply by batch size
batch = 64
for l in layers:
    l['flops'] *= batch
    l['bytes'] *= batch
    l['ai'] = l['flops'] / l['bytes']

# Plot roofline with layer annotations
fig, ax = plt.subplots(1, 1, figsize=(12, 7))

peak_flops = 200  # GFLOPS
bandwidth = 50    # GB/s
ridge = peak_flops / bandwidth

ai_range = np.logspace(-2, 3, 200)
roofline = np.minimum(peak_flops, bandwidth * ai_range)
ax.loglog(ai_range, roofline, 'b-', linewidth=3, label='Roofline', zorder=1)
ax.axvline(x=ridge, color='gray', linestyle='--', alpha=0.5)

# Plot layers
for l in layers:
    achieved_gflops = l['flops'] / 1e9 * 1000  # Assume 1ms execution → GFLOPS
    # For visualization, place at their AI with estimated performance
    perf = min(peak_flops * 0.6, bandwidth * l['ai'] * 0.8)  # 60-80% efficiency
    color = 'green' if l['ai'] > ridge else 'red'
    marker = 's' if 'MatMul' in l['name'] else 'o'
    ax.loglog(l['ai'], perf, marker, markersize=10, color=color, zorder=5)
    ax.annotate(l['name'], (l['ai'], perf), textcoords='offset points',
               xytext=(5, 5), fontsize=7, rotation=15)

ax.set_xlabel('Arithmetic Intensity (FLOP/byte)', fontsize=11)
ax.set_ylabel('Performance (GFLOPS)', fontsize=11)
ax.set_title('Roofline Analysis: Demo MLP Layers\n(Red=Memory-bound, Green=Compute-bound)', 
            fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, which='both')
ax.legend(fontsize=10)
ax.set_xlim([0.01, 1000])
ax.set_ylim([0.1, 500])

plt.tight_layout()
plt.savefig('perf_roofline.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table
print(f"\n{'Layer':<22} {'FLOPs':<12} {'Bytes':<10} {'AI':<8} {'Classification'}")
print("-" * 65)
for l in layers:
    bound = 'COMPUTE' if l['ai'] > ridge else 'MEMORY'
    print(f"{l['name']:<22} {l['flops']:<12,} {l['bytes']:<10,} {l['ai']:<8.2f} {bound}")

<a id='8'></a>
## 8. Benchmarking Methodology

### Statistical Rigor

A single timing measurement is meaningless. Proper benchmarking requires:

**Confidence Interval** for the mean latency:

$$\bar{x} \pm t_{\alpha/2, n-1} \cdot \frac{s}{\sqrt{n}}$$

Where:
- $\bar{x}$ = sample mean
- $t_{\alpha/2, n-1}$ = t-distribution critical value (1.96 for 95% CI with large n)
- $s$ = sample standard deviation
- $n$ = number of measurements

**Required sample size** for desired precision $\epsilon$:

$$n \geq \left(\frac{t_{\alpha/2} \cdot s}{\epsilon}\right)^2$$

### Percentile Reporting

For production systems, **tail latency** matters more than mean:

| Metric | Formula | Use |
|--------|---------|-----|
| P50 (median) | `sorted_times[n//2]` | Typical user experience |
| P95 | `sorted_times[int(0.95*n)]` | SLA target (most users) |
| P99 | `sorted_times[int(0.99*n)]` | Tail latency target |
| P99.9 | `sorted_times[int(0.999*n)]` | Worst case (1 in 1000) |

### Throughput Calculation

$$\text{Throughput} = \frac{N_{\text{samples}}}{t_{\text{total}}} \quad [\text{samples/sec}]$$

For batched inference:

$$\text{Throughput} = \frac{B}{T_{\text{batch}}} = \frac{B \cdot N_{\text{iterations}}}{\sum_{i=1}^{N} T_i}$$

### A/B Comparison

Use **paired t-test** or **Mann-Whitney U test** to determine if a change is statistically significant:

$$H_0: \mu_A = \mu_B \quad \text{vs} \quad H_1: \mu_A \neq \mu_B$$

$$t = \frac{\bar{d}}{s_d / \sqrt{n}}, \quad \text{where } d_i = x_{A,i} - x_{B,i}$$

Reject $H_0$ if $|t| > t_{\alpha/2, n-1}$ (p-value < 0.05).

### Common Pitfalls

```
┌─────────────────────────────────────────────────────────────────────┐
│              Benchmarking Anti-Patterns                               │
├─────────────────────────────────────────────────────────────────────┤
│                                                                      │
│  ✗ Timing first run    → Includes compilation, cache warming        │
│  ✗ Mean without stddev → Hides bimodal distributions                │
│  ✗ Battery/throttle    → Unstable clock frequency                   │
│  ✗ Shared machine      → Noisy neighbor interference                │
│  ✗ Small N             → Confidence interval too wide               │
│  ✗ Varying shapes      → Allocator noise dominates signal           │
│  ✗ No controlled vars  → Confounding factors                        │
│                                                                      │
│  ✓ Warmup 50+ runs     → Stable steady state                       │
│  ✓ 200+ timed runs     → Tight confidence intervals                │
│  ✓ Report percentiles  → Full distribution picture                  │
│  ✓ Pin CPU frequency   → Deterministic clocks                       │
│  ✓ Paired comparison   → Same conditions for A vs B                 │
│                                                                      │
└─────────────────────────────────────────────────────────────────────┘
```

In [ ]:
import onnxruntime as ort
import numpy as np
import time
from scipy import stats

def rigorous_benchmark(model_path, input_feed, n_warmup=50, n_iter=500,
                       providers=["CPUExecutionProvider"], session_options=None):
    """Production-grade benchmarking with statistical analysis."""
    so = session_options or ort.SessionOptions()
    sess = ort.InferenceSession(model_path, so, providers=providers)
    
    # Warmup
    for _ in range(n_warmup):
        sess.run(None, input_feed)
    
    # Timed runs
    times = []
    for _ in range(n_iter):
        t0 = time.perf_counter()
        sess.run(None, input_feed)
        times.append((time.perf_counter() - t0) * 1000)
    
    times = np.array(times)
    
    # Statistical analysis
    ci_95 = stats.t.interval(0.95, len(times)-1, loc=np.mean(times), scale=stats.sem(times))
    
    return {
        'mean': np.mean(times),
        'std': np.std(times),
        'p50': np.percentile(times, 50),
        'p95': np.percentile(times, 95),
        'p99': np.percentile(times, 99),
        'p999': np.percentile(times, 99.9),
        'ci_95_lower': ci_95[0],
        'ci_95_upper': ci_95[1],
        'cv': np.std(times) / np.mean(times),  # Coefficient of variation
        'n_samples': len(times),
        'raw_times': times,
    }

# Run benchmark
x = np.random.randn(32, 784).astype(np.float32)
result = rigorous_benchmark("perf_demo.onnx", {"X": x})

print("=" * 60)
print("RIGOROUS BENCHMARK REPORT")
print("=" * 60)
print(f"\n  Samples: {result['n_samples']}")
print(f"  Mean:    {result['mean']:.4f} ms")
print(f"  Std:     {result['std']:.4f} ms")
print(f"  CV:      {result['cv']:.4f} ({result['cv']*100:.1f}%)")
print(f"\n  Percentiles:")
print(f"    P50:   {result['p50']:.4f} ms")
print(f"    P95:   {result['p95']:.4f} ms")
print(f"    P99:   {result['p99']:.4f} ms")
print(f"    P99.9: {result['p999']:.4f} ms")
print(f"\n  95% Confidence Interval:")
print(f"    [{result['ci_95_lower']:.4f}, {result['ci_95_upper']:.4f}] ms")
print(f"    Width: {result['ci_95_upper'] - result['ci_95_lower']:.4f} ms")

# Throughput
batch_size = 32
throughput = batch_size / (result['p50'] / 1000)
print(f"\n  Throughput (at P50): {throughput:.0f} samples/sec")

<a id='9'></a>
## 9. End-to-End Tuning Workflow

### Systematic Optimization Strategy

```
┌─────────────────────────────────────────────────────────────────────────┐
│               Performance Tuning Workflow                                 │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                          │
│  1. MEASURE BASELINE                                                     │
│     └── Benchmark with default settings, record P50/P95/P99             │
│                                                                          │
│  2. IDENTIFY BOTTLENECK                                                  │
│     ├── Profile: which ops consume the most time?                       │
│     ├── Roofline: are hotspot ops compute or memory bound?              │
│     └── Transfer: is H2D/D2H overhead significant?                      │
│                                                                          │
│  3. APPLY OPTIMIZATION (in order of impact)                             │
│     ├── Graph opt level → ALL                                           │
│     ├── Thread tuning → match physical cores                            │
│     ├── Precision reduction → FP16/INT8 if accuracy allows              │
│     ├── IOBinding → if transfer > 10% of total                          │
│     ├── Batch size → find throughput-optimal B                          │
│     └── EP selection → try TensorRT, OpenVINO                           │
│                                                                          │
│  4. VALIDATE                                                             │
│     ├── Re-benchmark with same methodology                              │
│     ├── Verify numerical accuracy                                       │
│     └── Test under production-like load                                 │
│                                                                          │
│  5. ITERATE                                                              │
│     └── Re-profile, identify next bottleneck, repeat                    │
│                                                                          │
└─────────────────────────────────────────────────────────────────────────┘
```

### Decision Tree

$$\text{Next action} = \begin{cases}
\text{Enable graph opts} & \text{if level} < \text{ALL} \\
\text{Tune threads} & \text{if CPU util} < 80\% \\
\text{Quantize} & \text{if memory-bound and accuracy tolerant} \\
\text{IOBinding} & \text{if transfer} > 10\% \text{ of latency} \\
\text{Change EP} & \text{if GPU available and model large enough} \\
\text{Increase batch} & \text{if throughput matters more than latency}
\end{cases}$$

In [ ]:
import onnxruntime as ort
import numpy as np
import time

# Demonstrate a full tuning sweep
x = np.random.randn(32, 784).astype(np.float32)

configurations = [
    {"name": "Baseline (defaults)",
     "opt_level": ort.GraphOptimizationLevel.ORT_DISABLE_ALL,
     "intra": 1, "inter": 1},
    {"name": "+ Graph opt ALL",
     "opt_level": ort.GraphOptimizationLevel.ORT_ENABLE_ALL,
     "intra": 1, "inter": 1},
    {"name": "+ 4 intra-op threads",
     "opt_level": ort.GraphOptimizationLevel.ORT_ENABLE_ALL,
     "intra": 4, "inter": 1},
    {"name": "+ 8 intra-op threads",
     "opt_level": ort.GraphOptimizationLevel.ORT_ENABLE_ALL,
     "intra": 8, "inter": 1},
    {"name": "+ Memory arena+pattern",
     "opt_level": ort.GraphOptimizationLevel.ORT_ENABLE_ALL,
     "intra": 4, "inter": 1, "arena": True, "pattern": True},
]

print(f"{'Configuration':<30} {'P50(ms)':<10} {'P99(ms)':<10} {'Throughput':<12} {'Speedup':<10}")
print("-" * 75)

baseline_p50 = None
for cfg in configurations:
    so = ort.SessionOptions()
    so.graph_optimization_level = cfg["opt_level"]
    so.intra_op_num_threads = cfg["intra"]
    so.inter_op_num_threads = cfg["inter"]
    so.enable_cpu_mem_arena = cfg.get("arena", True)
    so.enable_mem_pattern = cfg.get("pattern", True)
    
    sess = ort.InferenceSession("perf_demo.onnx", so, providers=["CPUExecutionProvider"])
    
    for _ in range(30):
        sess.run(None, {"X": x})
    
    times = []
    for _ in range(300):
        t0 = time.perf_counter()
        sess.run(None, {"X": x})
        times.append((time.perf_counter() - t0) * 1000)
    
    p50 = np.median(times)
    p99 = np.percentile(times, 99)
    tput = 32 / (p50 / 1000)
    
    if baseline_p50 is None:
        baseline_p50 = p50
    speedup = baseline_p50 / p50
    
    print(f"{cfg['name']:<30} {p50:<10.4f} {p99:<10.4f} {tput:<12.0f} {speedup:<10.2f}x")

<a id='10'></a>
## 10. Performance Visualization & Analysis

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

# Comprehensive performance dashboard
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: A/B comparison with confidence intervals
np.random.seed(42)
times_A = np.random.lognormal(mean=np.log(1.2), sigma=0.15, size=500)
times_B = np.random.lognormal(mean=np.log(0.9), sigma=0.12, size=500)

axes[0,0].hist(times_A, bins=40, alpha=0.6, color='red', label=f'Config A (P50={np.median(times_A):.3f}ms)', density=True)
axes[0,0].hist(times_B, bins=40, alpha=0.6, color='blue', label=f'Config B (P50={np.median(times_B):.3f}ms)', density=True)

t_stat, p_value = stats.ttest_ind(times_A, times_B)
axes[0,0].set_title(f'A/B Latency Comparison\nt={t_stat:.2f}, p={p_value:.2e} (significant!)', fontweight='bold')
axes[0,0].set_xlabel('Latency (ms)')
axes[0,0].set_ylabel('Density')
axes[0,0].legend(fontsize=9)
axes[0,0].grid(True, alpha=0.3)

# Plot 2: Percentile ladder
percentiles = [50, 75, 90, 95, 99, 99.5, 99.9]
pct_A = [np.percentile(times_A, p) for p in percentiles]
pct_B = [np.percentile(times_B, p) for p in percentiles]

x = np.arange(len(percentiles))
width = 0.35
axes[0,1].bar(x - width/2, pct_A, width, label='Config A', color='red', alpha=0.7)
axes[0,1].bar(x + width/2, pct_B, width, label='Config B', color='blue', alpha=0.7)
axes[0,1].set_xticks(x)
axes[0,1].set_xticklabels([f'P{p}' for p in percentiles])
axes[0,1].set_xlabel('Percentile')
axes[0,1].set_ylabel('Latency (ms)')
axes[0,1].set_title('Percentile Ladder', fontweight='bold')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3, axis='y')

# Plot 3: Sample size vs CI width
sample_sizes = np.arange(10, 1001, 10)
ci_widths = 2 * stats.t.ppf(0.975, sample_sizes-1) * 0.15 / np.sqrt(sample_sizes)

axes[1,0].plot(sample_sizes, ci_widths * 1000, 'b-', linewidth=2)
axes[1,0].axhline(y=10, color='red', linestyle='--', label='Target precision: 0.01ms')
# Find where CI width < target
threshold_idx = np.argmax(ci_widths * 1000 < 10)
if threshold_idx > 0:
    axes[1,0].axvline(x=sample_sizes[threshold_idx], color='green', linestyle='--',
                     label=f'Min samples: {sample_sizes[threshold_idx]}')
axes[1,0].set_xlabel('Number of Samples')
axes[1,0].set_ylabel('95% CI Width (μs)')
axes[1,0].set_title('Required Samples for Precision\n$n \\geq (t_{0.025} \\cdot s / \\epsilon)^2$', fontweight='bold')
axes[1,0].legend(fontsize=9)
axes[1,0].grid(True, alpha=0.3)

# Plot 4: Throughput scaling with optimization
configs_labels = ['Baseline', '+GraphOpt', '+Threads', '+Fusion', '+IOBind']
throughputs = [5000, 7500, 15000, 22000, 28000]
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(configs_labels)))

bars = axes[1,1].bar(configs_labels, throughputs, color=colors, edgecolor='black', linewidth=0.5)
for bar, tput in zip(bars, throughputs):
    axes[1,1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                  f'{tput:,}', ha='center', fontsize=9, fontweight='bold')

axes[1,1].set_ylabel('Throughput (samples/sec)')
axes[1,1].set_title('Cumulative Optimization Impact', fontweight='bold')
axes[1,1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('perf_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

## Key Equations Summary

### Amdahl's Law
$$S(n) = \frac{1}{(1-p) + p/n}, \quad \lim_{n\to\infty} S(n) = \frac{1}{1-p}$$

### Memory Bandwidth
$$B = \frac{\text{data\_size}}{t_{\text{transfer}}}, \quad t_{\text{transfer}} = \frac{\text{data\_size}}{\text{BW}_{\text{effective}}}$$

### Roofline Model
$$\text{Perf} = \min(\text{Peak FLOPS}, \text{BW} \times \text{AI}), \quad \text{AI} = \frac{\text{FLOPs}}{\text{Bytes}}$$

### Throughput
$$\text{Throughput} = \frac{N_{\text{samples}}}{t_{\text{total}}} = \frac{B}{T(B)}$$

### Confidence Interval
$$\bar{x} \pm t_{\alpha/2, n-1} \cdot \frac{s}{\sqrt{n}}$$

### IOBinding Benefit
$$\text{Speedup}_{\text{IOB}} = \frac{T_{\text{copy}} + T_{\text{compute}} + T_{\text{sync}}}{T_{\text{compute}} + T_{\text{sync}}}$$

In [ ]:
# Cleanup
import os
for f in ['perf_demo.onnx', 'perf_fundamentals.png', 'perf_iobinding.png',
          'perf_roofline.png', 'perf_dashboard.png']:
    if os.path.exists(f):
        os.remove(f)
print("Cleanup complete.")

## Summary

Performance tuning ORT sessions follows a systematic methodology:

1. **Amdahl's Law** sets the theoretical ceiling — the serial fraction limits maximum speedup regardless of parallelization effort

2. **Roofline analysis** classifies operations as compute-bound (need more FLOPS) or memory-bound (need less data movement)

3. **Thread tuning** balances intra-op (within-operator) and inter-op (across-operator) parallelism without oversubscription

4. **Memory optimization** via arenas, patterns, and lifetime analysis reduces allocation overhead to near-zero

5. **Graph optimization** (fusion, folding) reduces kernel launches and memory traffic by up to 3x

6. **IOBinding** eliminates PCIe transfer overhead when data can stay on-device

7. **Statistical benchmarking** with proper warmup, sufficient samples, and confidence intervals ensures reliable measurements